In [1]:
import json
import os
import pandas as pd
import yfinance as yf
from datetime import date
from scipy.stats import gmean
from yfetch import delay, get_stock_history, get_stock_name, get_stock_metadata, to_weekly

risk_free_return = 4/100
history_weeks = 3 * 52 # 3Y
change_weeks = 52 # 1Y
range_window = 4 # ~ month
cache_days = 5 # how long to cache stock data
revenue_cache_file = 'data/revenue_cache.json'

symbols = [
    'MAGS',
    'FNGS',
    'IGM',
    'SPYG',
    'SPMO',
    'IWMO.MI',
    'TSLA',
    'NVDA',
    'AVGO',
    'SMH',
    'USD',
    'TDIV.AS',
    'ESIF.DE',
    'DFEN.DE',
    'EXI',
    '4GLD.DE',
    'BRK-B',
    'NFLX',
    'PLTR',
    'NET',
    'ISRG',
    'SHOP',
    'DAPP',
    'BITQ',
    'UFO',
    'ROKT',
    'JEDI.DE',
    'SPCX',
    'RKLB',
    'PL',
    'NASA',
    'NUKZ',
    'NLR',
    'AIPO',
    'QTUM',
    'AGIX',
    'CHAT',
    'BAI',
    'ARKG',
    'PPA',
    'QQQ',
    'COIN',
    'NKE',
    'STLA',
    'AMAT',
]

try:
    with open(revenue_cache_file) as f:
        revenue_cache = json.load(f)
except FileNotFoundError:
    revenue_cache = {}

def fetch_revenue_growth(symbol):
    """(year over year growth of the latest quarterly revenue, quarter end date),
    (None, None) if unavailable."""
    delay()
    qis = yf.Ticker(symbol).quarterly_income_stmt
    if 'Total Revenue' not in qis.index:
        print(f'No revenue for {symbol}')
        return None, None
    rev = qis.loc['Total Revenue'].dropna().sort_index(ascending=False)
    if len(rev) < 5:
        print(f'Not enough revenue history for {symbol}: {len(rev)} quarters, need 5')
        return None, None
    return round(float(rev.iloc[0] / rev.iloc[4] - 1), 4), str(rev.index[0].date())

def format_revenue(growth, quarter):
    return 'n/a' if growth is None else f'{growth:.2%} ({quarter})'

def get_revenue_growth(symbol):
    """Revenue growth from data/revenue_cache.json, refetched once it goes stale."""
    if get_stock_metadata(symbol).get('instrumentType') != 'EQUITY':
        return None # ETFs and the like have no revenue

    today = date.today()
    cached = revenue_cache.get(symbol)
    if cached and (today - date.fromisoformat(cached['fetched'])).days < cache_days:
        return cached['growth']

    growth, quarter = fetch_revenue_growth(symbol)
    if cached and (growth, quarter) != (cached['growth'], cached['quarter']):
        print(f'{symbol} revenue: {format_revenue(cached["growth"], cached["quarter"])}'
              f' => {format_revenue(growth, quarter)}')

    revenue_cache[symbol] = {'growth': growth, 'quarter': quarter, 'fetched': today.isoformat()}
    os.makedirs(os.path.dirname(revenue_cache_file), exist_ok=True)
    with open(revenue_cache_file, 'w') as f:
        json.dump(revenue_cache, f, indent=2, sort_keys=True)
    return growth

def temperature(series):
    """Share of past values at or below the last one, None if the series is empty."""
    series = series.dropna()
    if len(series) == 0:
        return None
    return (series <= series.iloc[-1]).mean()

def sma_temperature(symbol, daily, window=200):
    """Temperature of the distance to the SMA over daily history."""
    sma = daily.Close.rolling(window=window).mean()
    if sma.dropna().empty:
        print(f'Not enough history for {symbol}: {len(daily)} days, need {window}')
        return None
    return temperature(daily.Close / sma - 1)

rows = []
for symbol in symbols:
    # one daily fetch per symbol - the weekly bars are aggregated from it
    daily = get_stock_history(symbol, period='5y', interval='1d', cache_days=cache_days)
    history = to_weekly(daily, symbol).tail(history_weeks)

    if len(history) < history_weeks:
        print(f'Not enough history for {symbol}: {len(history)} weeks, need {history_weeks}')
        gmean_change = std = sharpe = None
    else:
        changes = history.Close.pct_change(periods=change_weeks, fill_method=None).dropna()
        gmean_change = gmean(1 + changes) - 1 # geometric mean of changes
        std = changes.std()
        sharpe = (gmean_change - risk_free_return) / std

    high = history.High.rolling(range_window).max()
    low = history.Low.rolling(range_window).min()
    range = ((high - low) / (high + low) * 2).dropna() # relative range per window

    rows.append({
        'symbol': symbol,
        'name': get_stock_name(symbol),
        'price': history.Close.iloc[-1], # last close
        'weeks': len(history),
        'gmean': gmean_change,
        'std': std,
        'sharpe': sharpe,
        'range': range.mean(),
        'revenue': get_revenue_growth(symbol),
        'T200': sma_temperature(symbol, daily, window=200),
    })

df = pd.DataFrame(rows)
f = f'data/folio.csv'
df.to_csv(f, index=False)
print(f'Saved to {f} ({len(df)} rows)')

df = df.reset_index(drop=True)
for col in ['gmean', 'std', 'range', 'revenue', 'T200']:
    df[col] = df[col].map(lambda v: None if pd.isna(v) else f'{v:.2%}')
df

Not enough history for SPCX: 13 weeks, need 156
Not enough history for SPCX: 56 days, need 200
Not enough history for NASA: 23 weeks, need 156
Not enough history for NASA: 107 days, need 200
Not enough history for NUKZ: 137 weeks, need 156
Not enough history for AIPO: 59 weeks, need 156
Not enough history for AGIX: 112 weeks, need 156
Not enough history for BAI: 98 weeks, need 156
Saved to data/folio.csv (45 rows)


,symbol,name,price,weeks,gmean,std,sharpe,range,revenue,T200
0,MAGS,Roundhill Magnificent Seven ETF,68.364998,156,33.20%,17.05%,1.712684,10.54%,None,22.36%
1,FNGS,MicroSectors FANG+ ETN,80.269997,156,30.64%,14.90%,1.787812,10.68%,None,54.55%
2,IGM,iShares Expanded Tech Sector ETF,159.065002,156,31.30%,14.45%,1.889614,9.93%,None,55.49%
3,SPYG,State Street SPDR Portfolio S&P 500 Growth ETF,119.860100,156,25.88%,9.62%,2.273624,7.78%,None,45.08%
4,SPMO,Invesco S&P 500 Momentum ETF,145.720001,156,33.68%,13.22%,2.244990,8.77%,None,53.22%
5,IWMO.MI,iShares Edge MSCI World Momentum Factor UCITS ...,98.050003,156,19.48%,13.88%,1.115215,7.66%,None,63.33%
6,TSLA,"Tesla, Inc.",356.540009,156,39.65%,30.63%,1.163874,23.02%,25.52%,30.87%
7,NVDA,NVIDIA Corporation,219.029999,156,60.77%,60.95%,0.931359,19.05%,85.23%,34.00%
8,AVGO,Broadcom Inc.,370.170013,156,73.80%,31.10%,2.244607,19.86%,47.87%,15.34%
9,SMH,VanEck Semiconductor ETF,545.770020,156,48.46%,42.96%,1.034725,14.91%,None,47.54%
